In [1]:
import time
from typing import List, Dict, Any, Optional
from elasticsearch import Elasticsearch, helpers
from FlagEmbedding import BGEM3FlagModel

class BGEElasticManager:
    def __init__(
        self, 
        model_path: str = "../model/bge-m3", 
        es_url: str = "http://localhost:9200",
        use_fp16: bool = True
    ):
        self.es = Elasticsearch(es_url)
        self.model = BGEM3FlagModel(model_name_or_path=model_path, use_fp16=use_fp16)

    def create_index(self, index_name: str, dims: int = 1024):
        """인덱스 생성 및 매핑 설정"""
        index_config = {
            "mappings": {
                "properties": {
                    # 계층화된 글로벌 메타데이터
                    "global_metadata": {
                        "properties": {
                            "title": {"type": "text", "analyzer": "standard"},
                            "ship_numbers": {"type": "keyword"},
                            "product_name": {"type": "keyword"},
                            "specifications": {"type": "text"},
                            "document_type": {"type": "keyword"}
                        }
                    },
                    "page_content": {"type": "text"},
                    "dense_vector": {
                        "type": "dense_vector",
                        "dims": dims,
                        "index": True,
                        "similarity": "cosine"
                    },
                    # rank_features는 점 표기법(.) 검색을 지원하지 않으므로 필드 정의 방식 주의
                    "sparse_tokens": {"type": "rank_features"}, 
                    "file_path": {"type": "keyword"},
                    "file_name": {"type": "keyword"},
                    "page_number": {"type": "integer"},
                }
            }
        }
        if self.es.indices.exists(index=index_name):
            print(f"Index '{index_name}' already exists.")
            return
        self.es.indices.create(index=index_name, body=index_config)
        print(f"Index '{index_name}' created successfully.")

    def _generate_actions(self, index_name: str, raw_documents: List[Any]):
        """Bulk 전송을 위한 Generator 구성"""
        contents = [doc.page_content for doc in raw_documents]
        
        # 1. Batch Encoding
        outputs = self.model.encode(
            contents, 
            return_dense=True, 
            return_sparse=True
        )
        
        dense_vecs = outputs.get('dense_vecs')
        lexical_weights = outputs.get('lexical_weights')

        for i, doc in enumerate(raw_documents):
            # 가중치가 0보다 큰 토큰만 필터링 (Rank Feature 저장용)
            sparse_dict = {str(k): float(v) for k, v in lexical_weights[i].items() if v > 0}
            
            # [수정포인트 1] metadata 내부에 global_metadata가 있다면 구조 유지
            # LangChain Document 등에서 metadata를 가져올 때의 구조를 고려합니다.
            source = {
                "page_content": doc.page_content,
                "dense_vector": dense_vecs[i].tolist(),
                "sparse_tokens": sparse_dict,
            }
            # 나머지 메타데이터(file_path, page_number 등)와 
            # 중첩된 global_metadata를 소스에 업데이트
            source.update(doc.metadata) 
            
            yield {
                "_index": index_name,
                "_source": source
            }

    def bulk_index(self, index_name: str, raw_documents: List[Any], batch_size: int = 64):
        """배치 단위로 인덱싱 실행"""
        start_time = time.time()
        total_docs = len(raw_documents)
        
        for i in range(0, total_docs, batch_size):
            batch = raw_documents[i:i + batch_size]
            # helpers.bulk는 generator를 직접 소비함
            helpers.bulk(self.es, self._generate_actions(index_name, batch))
            print(f"Progress: {min(i + batch_size, total_docs)}/{total_docs} indexed.")

        print(f"Indexing completed in {time.time() - start_time:.2f}s")

    def hybrid_search(
        self, 
        index_name: str, 
        query_text: str, 
        ship_number: Optional[str] = None,  # 필터링할 호선 번호 추가
        top_k: int = 5, 
        dense_weight: float = 1.0, 
        sparse_weight: float = 1.0
    ):
        """Dense + Sparse + Metadata Filter 하이브리드 검색"""
        
        # 1. 쿼리 임베딩
        query_output = self.model.encode([query_text], return_dense=True, return_sparse=True)
        query_dense = query_output['dense_vecs'][0].tolist()
        query_sparse = {str(k): float(v) for k, v in query_output['lexical_weights'][0].items() if v > 0}

        # 2. Sparse (Rank Feature) 쿼리 구성
        actual_sparse_queries = [
            {
                "rank_feature": {
                    "field": f"sparse_tokens.{token}", 
                    "boost": float(weight * sparse_weight)
                }
            }
            for token, weight in query_sparse.items()
        ]

        # 3. 메인 쿼리 구성
        search_query = {
            "size": top_k,
            "query": {
                "bool": {
                    "should": actual_sparse_queries,
                    "filter": []  # 필터 조건 리스트
                }
            },
            "knn": {
                "field": "dense_vector",
                "query_vector": query_dense,
                "k": top_k,
                "num_candidates": 100,
                "boost": dense_weight
            }
        }

        # 4. 호선 번호 필터 추가 (값이 있을 경우에만)
        if ship_number:
            search_query["query"]["bool"]["filter"].append({
                "term": {
                    "global_metadata.ship_numbers": ship_number
                }
            })
            # kNN 검색에도 필터를 적용해야 정확한 top_k 결과가 나옵니다.
            search_query["knn"]["filter"] = {
                "term": {
                    "global_metadata.ship_numbers": ship_number
                }
            }

        response = self.es.search(index=index_name, body=search_query)
        
        return [
            {
                "score": hit["_score"],
                "content": hit["_source"].get("page_content"),
                "metadata": {k: v for k, v in hit["_source"].items() if k not in ["dense_vector", "sparse_tokens"]}
            }
            for hit in response['hits']['hits']
        ]

d:\auto_vectordb\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: Could not import module 'TrainingArguments'. Are this object's requirements defined correctly?

In [ ]:
bem= BGEElasticManager()
bem

In [ ]:
bem.create_index(index_name="pos")

In [ ]:
import pickle
with open("./docs/FWG_with_global.pkl", "rb") as f:
    loaded_text2 = pickle.load(f)
len(loaded_text2)

In [ ]:
bem.bulk_index(index_name="pos", raw_documents=loaded_text2)

In [ ]:
bem.hybrid_search(index_name="pos", query_text="장비 주요 사양 요약")